<a id="unit-u.b0e614ac73e14215bced76cf33ad4bda"></a>

<div>
  <div>6-ЗЕРТХАНАЛЫҚ ЖҰМЫС · APACHE DORIS КӨМЕГІМЕН ДЕРЕКТЕР ҚОЙМАСЫН ҚҰРУ</div>
  <div>Кірісті сақтау, қабылданбағандарды бағыттау және тексеруді автоматтандыру</div>
</div>

Он үш жаңа тапсырыс келеді, үшеуі қате: сан емес сома, тапсырыс ID-і жоқ жол және жоқ тұтынушы. Stream Load арқылы он үшін де мәтін түрінде жүктеп, SQL ережелерімен әрқайсысын қабылданған не қабылданбаған кестеге бағыттайсыз, әр қабылдамауды бастапқы кірісіне дейін қадағалайсыз және деректерді әдейі бұзып, тексерулерді сынайсыз. 7-модуль он қабылданған тапсырыстан басталады. Шамамен 30 минут жоспарлаңыз.

**6-модуль: деректер сапасы және схеманы тексеру** · [Сабақ](course.md) · [Тест](quiz06_data_quality_and_rejection.ipynb) · [Курстың README файлы](../../README.md)


<a id="unit-u.0cf5469755fd47848200ebfd49784e82"></a>

<a id="lab-scope"></a>

## Зертханалық жұмыстың қамту аясы

Бұл жұмыс `orders_raw`, `customers`, `orders_classified` көрінісін, `orders_clean` және `orders_rejected` кестелерін қайта жасайды. 7-модуль `orders_clean` және `customers` оқиды.

Тұтынушы өлшемі 5-зертханалық жұмыс жүктейтін `wwi_customers` кестесінен алынады (663 жол). 5-зертханалық жұмыс бұл дерекқорда орындалмаса, баптау ұяшығы сол бір кестені commit-ке енгізілген WWI Parquet пакетінен жүктейді; бар `wwi_customers` өзгеріссіз қайта пайдаланылады, ал жол саны қате көшірме ештеңе қалпына келтірілмей тұрып жұмысты тоқтатады.


In [ ]:
from pathlib import Path
import sys
import os
from uuid import uuid4

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import COURSE_ROOT, fixture, expect, expected_failure, normalized
from dw_course.schema import ORDER_COLUMNS, order_ddl, order_rows
from dw_course.wwi import manifest, parquet_ddl, parquet_paths
from dw_course.ui import show_sql, show_response, show_records

lab = connect_sandbox(module=6)

<a id="unit-u.014f149e3cb8476eac4ccbec7b49ceeb"></a>

<a id="1-stage-the-raw-input-with-stream-load"></a>

## 1. Бастапқы кірісті Stream Load арқылы аралық кестеге сақтау

`datasets/raw_orders.json` он үш кірісті бір JSON массиві ретінде сақтайды. Әр нысанда осы келуді анықтайтын `input_id` және дереккөз жіберген тапсырыс өрістері бар: 11-кірісте `"order_amount": "not-a-number"`, 12-кірісте `"order_id": null`, 13-кірісте `"customer_id": "999999"`.

`orders_raw` әр бизнес бағанын `VARCHAR` ретінде сақтайды. Мәтін бағаны кез келген мәтінді қабылдайды, сондықтан 5-зертханалық жұмыстағы `strict_mode` және `max_filter_ratio` = 0 параметрлері бар Stream Load он үш жолдың ешқайсысын қабылдамай тастамайды, сан емес мән дәл келген күйінде қалады. Түрлері қатаң берілген кесте қате жолдарды кіреберісте қабылдамай, мазмұны жоғалар еді. JSON үшін Stream Load әр нысан кілтін `columns` ішіндегі атауларға сәйкестендіреді; `strip_outer_array` файлдың нысандардан тұратын бір массив екенін айтады.

`customers` тұтынушы өлшемі — `wwi_customers` кестесінен көшірілген Unique Key кестесі: осы кірістерден жасалған емес, нақты тұтынушылардың тәуелсіз тізімі.


In [ ]:
lab.execute("DROP VIEW IF EXISTS orders_classified")
lab.execute("DROP TABLE IF EXISTS orders_raw")
fields = ",\n    ".join(col + " VARCHAR(100) NULL" for col in ORDER_COLUMNS)
raw_ddl = f'''CREATE TABLE orders_raw (
    input_id BIGINT NOT NULL,
    {fields}
) DUPLICATE KEY(input_id)
DISTRIBUTED BY HASH(input_id) BUCKETS 1
PROPERTIES ("replication_num"="1")'''
show_sql("The raw table: every business column is text", raw_ddl)
lab.execute(raw_ddl)
raw_columns = ",".join(["input_id", *ORDER_COLUMNS])
raw_label = "raw_" + uuid4().hex
show_sql("Stream Load request for the raw input", f"""PUT {os.environ["DW_BE_HTTP_URL"]}/api/{lab.database}/orders_raw/_stream_load
label: {raw_label}
format: json
strip_outer_array: true
columns: {raw_columns}
strict_mode: true
max_filter_ratio: 0
group_commit: off_mode
Request body: datasets/raw_orders.json""")
response = lab.stream_load("orders_raw", COURSE_ROOT / "datasets/raw_orders.json", raw_label, raw_columns, format="json")
show_response(response, title="Raw input Stream Load response")
expect((response["Status"], response["NumberLoadedRows"], response["NumberFilteredRows"]), ("Success", 13, 0),
       title="All thirteen inputs are staged")
lab.sql("SELECT input_id, order_id, customer_id, order_amount, data_source FROM orders_raw WHERE input_id > 9 ORDER BY input_id",
        title="The last four inputs, as they arrived");

lab.execute("DROP TABLE IF EXISTS customers")
lab.execute('CREATE TABLE customers (customer_id BIGINT NOT NULL, customer_name STRING) UNIQUE KEY(customer_id) DISTRIBUTED BY HASH(customer_id) BUCKETS 1 PROPERTIES("replication_num"="1")')
if not lab.query("SHOW TABLES LIKE 'wwi_customers'"):
    paths = parquet_paths()
    metadata = manifest()["tables"]["customers"]
    lab.execute(parquet_ddl("customers", "wwi_customers"))
    response = lab.stream_load(
        "wwi_customers", paths["customers"], "module6_wwi_" + uuid4().hex, format="parquet"
    )
    show_response(response)
    expect(response["Status"], "Success")
    expect(response["NumberLoadedRows"], metadata["rows"])
    expect(response["NumberFilteredRows"], 0)

lab.execute("INSERT INTO customers SELECT CustomerID, CustomerName FROM wwi_customers")
expect(lab.query("SELECT COUNT(*) FROM customers"), [(663,)], title="The customer dimension has 663 customers")

<a id="unit-u.e5848af9b7fa448aac69be6c127d2c54"></a>

**Күтілетін нәтиже**

Кесте анықтамасы, сұрау және жауабы: `Status` Success, `NumberLoadedRows` 13, `NumberFilteredRows` 0 және жасыл белгі. Соңғы төрт кіріс мәселелерді мәтін ретінде көрсетеді: 10-кіріс — қалыпты тапсырыс (900010, 10-тұтынушы, 250.00), 11-кірістің сомасы `not-a-number`, 12-кірісте тапсырыс ID-і жоқ (`NULL`), 13-кірісте `999999` тұтынушысы бар. Төртеуі де `COURSE_SIMULATION`.

Содан кейін 663 тұтынушыға арналған жасыл белгі. 5-зертханалық жұмыс осы дерекқорда орындалмаса, ұяшық алдымен `wwi_customers` жүктеп, сол жүктеудің 663 жолдық жауабын көрсетеді.


<a id="unit-u.41a6fb6f25b74e43afff0589510b3845"></a>

<a id="2-route-records-using-fixed-rules"></a>

## 2. Жазбаларды тұрақты ережелермен бағыттау

Жіктеу көрінісі TRY_CAST арқылы өрістерді түрлендіруге тырысып, тапсырыс ID-ін, соманы, тұтынушы сілтемесін және дереккөзді ретімен тексереді, алғашқы сәйкес қабылдамау себебін тіркейді. Содан кейін қабылданған жолдарды orders_clean ішіне, мәселелі input_id мәндері мен себептерін orders_rejected ішіне жазыңыз.

Он қабылданған және үш қабылданбаған жол күтіңіз; әр қатенің бастапқы мәтінін көру үшін orders_raw кестесіне input_id бойынша біріктіріңіз.


In [ ]:
lab.execute("""
CREATE VIEW orders_classified AS
SELECT *, CASE
    WHEN TRY_CAST(order_id AS BIGINT) IS NULL THEN 'INVALID_ORDER_ID'
    WHEN TRY_CAST(order_amount AS DECIMAL(12,2)) IS NULL
      OR TRY_CAST(order_amount AS DECIMAL(12,2)) < 0 THEN 'INVALID_AMOUNT'
    WHEN TRY_CAST(customer_id AS BIGINT) IS NULL THEN 'INVALID_CUSTOMER'
    WHEN TRY_CAST(customer_id AS BIGINT) NOT IN (SELECT customer_id FROM customers) THEN 'INVALID_CUSTOMER'
    WHEN data_source IS NULL OR data_source <> 'COURSE_SIMULATION' THEN 'INVALID_SOURCE'
    ELSE NULL END AS reject_reason
FROM orders_raw
""")
lab.execute("DROP TABLE IF EXISTS orders_clean")
ddl = order_ddl("orders_clean")
show_sql("Table creation SQL", ddl)
lab.execute(ddl)
lab.execute("DROP TABLE IF EXISTS orders_rejected")
lab.execute('CREATE TABLE orders_rejected (input_id BIGINT, reason VARCHAR(32)) DUPLICATE KEY(input_id) DISTRIBUTED BY HASH(input_id) BUCKETS 1 PROPERTIES("replication_num"="1")')
lab.execute(f"INSERT INTO orders_clean ({','.join(ORDER_COLUMNS)}) SELECT {','.join(ORDER_COLUMNS)} FROM orders_classified WHERE reject_reason IS NULL")
lab.execute("INSERT INTO orders_rejected SELECT input_id, reject_reason FROM orders_classified WHERE reject_reason IS NOT NULL")
lab.sql("SELECT reject_reason, COUNT(*) AS input_rows FROM orders_classified GROUP BY reject_reason ORDER BY reject_reason", title="Input routing results");
lab.sql("SELECT r.input_id, r.order_id, r.order_amount, r.customer_id, x.reason FROM orders_rejected x JOIN orders_raw r ON x.input_id=r.input_id ORDER BY r.input_id", title="Three rejected records and their raw fields");

<a id="unit-u.3ca3e87f7858412d8890eec37a45bdd9"></a>

**Күтілетін нәтиже**

`orders_clean` анықтамасы, содан кейін бағыттау нәтижесі: себебі жоқ (`NULL`) 10 кіріс, `INVALID_AMOUNT`, `INVALID_CUSTOMER` және `INVALID_ORDER_ID` себептерінің әрқайсысына бір кіріс. Қабылданбаған үш жазба бастапқы мәтініне бірігеді: 11-кірісте `not-a-number`, 12-кірісте тапсырыс ID-і жоқ, 13-кірісте `999999` тұтынушысы бар.

`999999` бүтін санға оңай түрленеді: түрі дұрыс. Ол `customers` ішінде 999999 тұтынушысы жоқ болғандықтан қабылданбайды, бұл — баған түрі білдіре алмайтын бизнес ережесі.


<a id="unit-u.0d55bcd0fa3e4369a3619e85d312acba"></a>

<a id="3-check-business-quality-and-inject-errors"></a>

## 3. Бизнес сапасын тексеру және қателер енгізу

Алдымен он үш кірістің бәрі бағытталғанын, содан кейін тапсырыс бірегейлігін, тұтынушы сілтемелерін, сомаларды, күйлерді және толық жазбаларды тексеріңіз. Оқиға уақыттарын осы үлгі пакетінің бекітілген шегімен салыстырыңыз.

Содан кейін агрегаттық тексерудің сәтсіз болуын күтпей, әр жолы бірегейлік тексеруін тікелей шақырып, екі қарсы мысалды сынаңыз:

1. Қайталанған тапсырыс қосып, қайталанған тапсырыс ID-ін сұрап, көрсетіңіз.
2. Деректерді қалпына келтіргеннен кейін тек екінші жолдың тапсырыс ID-ін біріншісінің ID-іне өзгертіп, басқа өрістерді өзгертпеңіз. Әлі жалпы сомасы 1400.00 болатын он жол бар, бірақ бір тапсырыс ID-і қайталанады, басқа тапсырыс жоқ.

Бірегейлік тексеруі екі қатені де табуы керек. Әр тәжірибеден кейін қабылданған кестені сақталған жіктелген кірістен қалпына келтіріп, 7-модульге бермес бұрын барлық өрісті қайта тексеріңіз.


In [ ]:
expected_rows = order_rows(fixture("orders.json"))
def check_unique_orders():
    from dw_course.ui import show_records
    actual = lab.query("SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM orders_clean")
    show_records("Uniqueness check details", [{"Check": "Duplicate order IDs", "Expected": 0,
                 "Actual": actual[0][0], "Result": "PASS" if actual == [(0,)] else "FAIL"}], collapsed=True)
    expect(actual, [(0,)])

def restore_clean_orders():
    lab.execute("TRUNCATE TABLE orders_clean")
    lab.execute(f"INSERT INTO orders_clean ({','.join(ORDER_COLUMNS)}) SELECT {','.join(ORDER_COLUMNS)} FROM orders_classified WHERE reject_reason IS NULL")

def quality_report():
    # DATE_FORMAT makes the transport representation explicit for fixture comparison.
    projection = ",".join("DATE_FORMAT(event_time, '%Y-%m-%d %H:%i:%s')" if col == "event_time" else col for col in ORDER_COLUMNS)
    checks = [
        ("Rows and amount", lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_clean"), [(10,"1400.00")]),
        ("Duplicate order IDs", lab.query("SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM orders_clean"), [(0,)]),
        ("NULL events or after 2026-01-02 12:00:00", lab.query("SELECT COUNT(*) FROM orders_clean WHERE event_time > '2026-01-02 12:00:00' OR event_time IS NULL"), [(0,)]),
        ("Not CREATED or already paid/refunded", lab.query("SELECT COUNT(*) FROM orders_clean WHERE status <> 'CREATED' OR paid_amount <> 0 OR refund_amount <> 0"), [(0,)]),
        ("Unknown customers", lab.query("SELECT COUNT(*) FROM orders_clean o LEFT JOIN customers c ON o.customer_id=c.customer_id WHERE c.customer_id IS NULL"), [(0,)]),
        ("Wrong data source", lab.query("SELECT COUNT(*) FROM orders_clean WHERE data_source <> 'COURSE_SIMULATION'"), [(0,)]),
        ("Every field matches the expected orders", normalized(lab.query(f"SELECT {projection} FROM orders_clean ORDER BY order_id")) == normalized(expected_rows), True),
    ]
    show_records("Quality report: expected and actual for every check", [
        {"Check": name, "Expected": normalized(expected), "Actual": normalized(actual),
         "Result": "PASS" if normalized(actual) == normalized(expected) else "FAIL"}
        for name, actual, expected in checks
    ], collapsed=True)
    for name, actual, expected in checks:
        expect(actual, expected)
    expect(True, True, title="Quality report: ten unique orders, 1400.00, new, known customers, every field as expected")

expect(lab.query("SELECT input_id, reason FROM orders_rejected ORDER BY input_id"),
       [(11,"INVALID_AMOUNT"),(12,"INVALID_ORDER_ID"),(13,"INVALID_CUSTOMER")], title="Three rejections with their reasons")
expect(lab.query("SELECT COUNT(*) FROM orders_classified WHERE reject_reason IS NULL"), [(10,)], title="Ten inputs pass every rule")
expect(lab.query("SELECT COUNT(*) FROM orders_raw r LEFT JOIN orders_rejected x ON r.input_id=x.input_id WHERE x.input_id IS NULL"), [(10,)],
       title="Every input is either accepted or rejected")
quality_report()

# Trigger the uniqueness check separately so aggregate checks do not interrupt it first.
lab.insert("orders_clean", ORDER_COLUMNS, [expected_rows[0]])
lab.sql("SELECT order_id, COUNT(*) AS copies FROM orders_clean GROUP BY order_id HAVING COUNT(*) > 1 ORDER BY order_id", title="Duplicate order ID after appending");
with expected_failure("Duplicate order check", "Duplicate order detected; the uniqueness rule works"):
    check_unique_orders()
restore_clean_orders()
quality_report()

# A subtler error: the second row uses the first row's order ID, leaving the row count and amount unchanged.
probe_rows = list(expected_rows)
probe_rows[1] = (probe_rows[0][0], *probe_rows[1][1:])
lab.execute("TRUNCATE TABLE orders_clean")
lab.insert("orders_clean", ORDER_COLUMNS, probe_rows)
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_clean"), [(10,"1400.00")], title="The totals still look right: ten rows, 1400.00")
lab.sql("SELECT order_id, COUNT(*) AS copies FROM orders_clean GROUP BY order_id HAVING COUNT(*) > 1 ORDER BY order_id", title="Correct totals, but a duplicate order ID");
with expected_failure("Duplicate check with unchanged totals", "The row count and amount are correct, but the uniqueness check still detects the error"):
    check_unique_orders()
restore_clean_orders()
quality_report()
lab.sql("SELECT COUNT(*) AS orders, SUM(order_amount) AS amount FROM orders_clean", title="Accepted orders restored after both error experiments");

<a id="unit-u.522c986f1e0d4536a3e1564343570e67"></a>

**Күтілетін нәтиже**

Бағыттауға арналған үш жасыл белгі, содан кейін он қабылданған тапсырыс пен 1400.00 сомасын, қайталанған тапсырыс ID-і жоқтығын, әр оқиғаның `2026-01-02 12:00:00` шегінен бұрын екенін, бәрі CREATED және төленген не қайтарылған сома жоқтығын, әр тұтынушы барын, әр тапсырыстың модельдеуден келгенін және әр өрістің күтілетін тапсырысқа теңдігін тексеретін сапа есебінің бір карточкасы.

Әр тексеру атауын, күтілетін және нақты мәндерін, PASS/FAIL нәтижесін көру үшін **Quality report: expected and actual for every check** ашыңыз. Әр әдейі қате үшін шағын **Uniqueness check details** кестесі де бар: онда тек сол ереже бағаланады, сондықтан алдымен жалпы сома тексеруінің сәтсіздігі бірегейлік қатесін жасыра алмайды.

Содан кейін екі тәжірибе. Көшірме қосылғаннан кейін 900001 тапсырысының 2 көшірмесі бар және бірегейлік тексеруі сәтсіз, жасыл карточкамен көрсетіледі: бұл күтілетін сәтсіздік. Деректер қалпына келтіріліп, есеп қайта өтеді. Екінші тәжірибеде он жолдың сомасы әлі 1400.00, сондықтан қорытындылар дұрыс көрінеді, бірақ 900001 екі рет, 900002 жоқ, бірегейлік тексеруі қайта сәтсіз. Екінші қалпына келтіруден кейін соңғы кесте 10 тапсырыс және 1400.00 көрсетеді: 7-модуль оқитын күй.


<a id="unit-u.07216a63df5e4848b9a7f5d346fc0e9a"></a>

<a id="4-practice-try-a-new-rule-before-you-use-it"></a>

## 4. Жаттығу: жаңа ережені қолданбас бұрын сынау

Бизнес ереже қосады: қабылданған тапсырыстың сомасы 200-ден жоғары болса, қолмен қарау қажет; басқалары қабылданады. Қабылданған тапсырыстарды саны және сомасымен `ACCEPT` пен `REVIEW` топтарына бөлетін `CASE` бар `SELECT` жазыңыз. Қабылдауға 8 тапсырыс және 850.00, қарауға 2 тапсырыс және 550.00 күтіледі. Мұнда ережені тек сынаңыз: 7-модуль үшін `orders_clean` өзгеріссіз қалсын.


In [ ]:
# Your code here

<a id="unit-u.82388918cfbd4e00b7e794c72d9aa828"></a>

Келесі ұяшықта үлгі шешім бар; ол жиналған күйде. Салыстыру үшін ашыңыз немесе орындаңыз.


In [ ]:
query = """SELECT CASE WHEN order_amount > 200 THEN 'REVIEW' ELSE 'ACCEPT' END AS decision,
COUNT(*) AS orders, SUM(order_amount) AS amount
FROM orders_clean GROUP BY decision ORDER BY decision"""
lab.sql(query, title="New amount review rule");
expect(lab.query(query), [("ACCEPT",8,"850.00"),("REVIEW",2,"550.00")], title="Eight to accept, two to review")
# Only try out the new rule; preserve the ten accepted orders used by Module 7.
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_clean"), [(10,"1400.00")], title="orders_clean is unchanged")

<a id="unit-u.2a8bf88a3dba4918b3c4e83ec72ea1ed"></a>

**Күтілетін нәтиже**

Екі жол: 8 тапсырыс және 850.00 бар `ACCEPT`, 2 тапсырыс және 550.00 бар `REVIEW`, екі жасыл белгі, екіншісі `orders_clean` ішінде әлі он тапсырыс және 1400.00 барын көрсетеді. Сынақ `SELECT` ештеңені өзгертпейді; ережені қолдану үшін оны жіктеу көрінісіне қосып, қайта бағыттар едіңіз.


<a id="unit-u.c786c68c5d854c868ab98fecd5f4df35"></a>

<a id="you-are-done-when"></a>

## Жұмысты аяқтау шарттары

- он үш кірістің бәрі `orders_raw` ішінде сақталған, он кіріс `orders_clean` ішінде (1400.00), үшеуі `orders_rejected` ішінде, әрқайсысын `input_id` арқылы бастапқы мәтініне дейін қадағалауға болады;
- `999999` тұтынушысының бүтін санға түрленіп, бәрібір қабылданбау себебін түсіндіре аласыз;
- екі белгілі қатеде, соның ішінде жалпы сомалары дұрыс болған біреуінде бірегейлік тексеруінің сәтсіз болуын және әр қалпына келтіруден кейін өтуін көрдіңіз.


<a id="unit-u.1349f7243389400ca3a656e67ed9af57"></a>

<a id="next"></a>

## Келесі қадам

[6-тест: деректер сапасы және схеманы тексеру](quiz06_data_quality_and_rejection.ipynb) файлын ашыңыз, содан кейін [7-модуль: деректерді жаңарту, жою және оқиғаларды қайта ойнату](../module07-state-changes/course.md) бөліміне өтіңіз.
